# 02 – Simulationspipeline (synthetische Fußball-Datensätze)

Erzeugt einen **nicht-sensiblen** synthetischen Datensatz,
in genau dem **`long_df`-Schema**, damit später
**dieselbe ACF-Funktion** unverändert auf echten und simulierten Daten läuft.

Als Spielplan $G_m^P$ wird der reale, frei verfügbare Fixture-Plan
`PremierLeague_2627.csv` verwendet (nur Paarungen/Spieltage, **keine Ergebnisse**).

**Ablauf:** 0. Setup & Parameter (Berechnung der Varianz in Notebook `calculate_variance.ipynb`) · 1. Leistungsstärke $S_{i,m}$ · 2. Spielplan laden ·
3. Tordifferenz & Poisson · 4. Saison → long_df · 5. Ausführen · 6. Validierung ·
7. Plots · 8. Speichern

Die Reproduzierbarkeit wird durch die Bereitstellung der simulierten Datensätze gewährleistet.

## 0. Setup & Parameter

In [6]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt # Für ACF-Plots
from pathlib import Path

def find_repo_root(start=None):
    """Sucht aufwaerts nach dem Repo-Wurzelverzeichnis, damit Pfade unabhaengig
    vom Arbeitsverzeichnis (Repo-Root oder notebooks/) stimmen.
    Marker: .git (nach jedem clone/pull vorhanden, nur im Repo-Root) oder
    README.md -- funktioniert also auch fuer andere Nutzer, die das Repo pullen."""
    p = Path(start or Path.cwd()).resolve()
    for cand in [p, *p.parents]:
        if (cand / ".git").exists() or (cand / "README.md").exists():
            return cand
    return p

REPO_ROOT = find_repo_root()

# --- Parameter ---
V            = 0.5071   # v = Var(S_i), aus den Daten geschaetzt
SIGMA        = 2.7075     # durchschnittliche Torsumme pro Spiel  (aus Daten berechnet)
ALPHA        = 0.9      # AR(1)-Koeffizient (nur Modell "ar1"). Default-Wert.
LAMBDA_FLOOR = 0.1      # Abfangregel fuer negatives lambda. TODO: Zählen wie oft dieser Fall eintritt. 
BASE_DATE    = pd.Timestamp("2026-08-01")   # Platzhalter-Startdatum (siehe long_df-Hinweis)
RHO = 0.5               # Default-Wert. Verschiedene Werte durchlaufen --> siehe Word "Notizen BA".

## 1. Leistungsstaerke $S_{i,m}$

Vier Varianten. `S` hat die Form `(n_teams, n_matchdays)`. Nach jeder Simulation wird
$\sum_i S_{i,m}=0$ je Spieltag erzwungen.

In [7]:
# TODO: Varianz wird durch das Zentrieren etwas unterschätzt. Laut Claude: Var(S_i gemittelt) = v * (1 - 1/n_teams). Bei 20 Teams also ca. 5% Unterschätzung. THEMATISIEREN!
def center_per_matchday(S):
    """Konstruktions-Invariante  Sum_i S_{i,m} = 0  je Spieltag."""
    return S - S.mean(axis=0, keepdims=True)

def simulate_strength_constant(n_teams, n_matchdays, v, rng):
    """(i) Konstante Staerke: S_i ~ N(0, v), ueber alle m gleich, kein Rauschen."""
    # Spalte wird einmal gezogen und dann kopiert.
    S_i = rng.normal(0.0, np.sqrt(v), size=n_teams)
    S = np.repeat(S_i[:, None], n_matchdays, axis=1)
    return center_per_matchday(S)

def simulate_strength_random(n_teams, n_matchdays, v, rng):
    """(ii) Zufaellige Staerke: S_{i,m} ~ N(0, v) unabhaengig je Spieltag m."""
    # Ganze Matrix wird direkt gezogen.
    S = rng.normal(0.0, np.sqrt(v), size=(n_teams, n_matchdays))
    return center_per_matchday(S)

def ar1_sigma_eps(v, alpha):
    """Zwingende AR(1)-Stationaritaetsbeziehung:  sigma_eps^2 = v (1 - alpha^2)."""
    return np.sqrt(v * (1.0 - alpha**2))

def simulate_strength_ar1(n_teams, n_matchdays, v, alpha, rng):
    """(iii) AR(1): S_{i,m} = alpha*S_{i,m-1} + eps,  eps ~ N(0, v(1-alpha^2)).
    Startwert S_{i,0} ~ N(0, v); alpha < 1 (Regression zur Mitte)."""
    # Bei alpha >= 1 würde die Varianz explodieren.
    if not 0.0 <= alpha < 1.0:
        raise ValueError("alpha muss in [0, 1) liegen (Stationaritaet).")
    sigma_eps = ar1_sigma_eps(v, alpha)
    S = np.empty((n_teams, n_matchdays))
    # Startwerte.
    S[:, 0] = rng.normal(0.0, np.sqrt(v), size=n_teams)
    for m in range(1, n_matchdays):
        S[:, m] = alpha * S[:, m - 1] + rng.normal(0.0, sigma_eps, size=n_teams)
    return center_per_matchday(S)

def simulate_strength_constant_plus_form(n_teams, n_matchdays, v, rho, rng):
    """(iv) Konstante Basis-Staerke + weisses Tagesform-Rauschen.
    S_{i,m} = mu_i + f_{i,m},  
        mu_i ~ N(0, rho*v) Grundstaerke,
        f_{i,m} ~ N(0, (1-rho)*v) iid je Spieltag. 
    v = Var(S_i) = Var(mu_i) + Var(f_{i,m}) = rho*v + (1-rho)*v = v. (Stationaritaet)
    Grenzfaelle: rho=1 -> konstant (i),  rho=0 -> random (ii)."""
    if not 0.0 <= rho <= 1.0:
        raise ValueError("rho muss in [0, 1] liegen.")
    mu_i = rng.normal(0.0, np.sqrt(rho * v),       size=n_teams)                    # einmalige Basis
    f    = rng.normal(0.0, np.sqrt((1.0 - rho) * v), size=(n_teams, n_matchdays))   # Tagesform
    S = mu_i[:, None] + f
    return center_per_matchday(S)

## 2. Spielplan $G_m^P$ laden

Realer, frei verfuegbarer Fixture-Plan der Premier League Saison 2026/27
(Quelle: https://www.premierleague.com/en/matches/premier-league/2026-27/matchweek-1).
Team-IDs werden auf zusammenhaengende, 0-basierte Indizes abgebildet; `N_TEAMS` und die
Zahl der Spieltage ergeben sich aus der Datei. Da jedes Team pro Spieltag genau einmal
spielt, ist `match_number = matchday_index (0-basiert)`.

In [8]:
SCHEDULE_PATH = REPO_ROOT / "data" / "schedule" / "PremierLeague_2627.csv"

def load_schedule(path):
    """Laedt einen realen Fixture-Plan (nur Paarungen/Spieltage, keine Ergebnisse).
    Rueckgabe: (schedule, team_names, n_teams).
    Setzt voraus, dass die Team-IDs bereits 0-basiert und lueckenlos sind
    (0 .. n_teams-1) -- erfuellt fuer PremierLeague_2627.csv.
      schedule    Liste von Spieltagen mit (heim_idx, gast_idx)-Paaren, 0-basiert
      team_names  dict  idx -> Teamname
      n_teams     Anzahl Teams
    """
    df = pd.read_csv(path)
    ids = sorted(set(df["home_id"]) | set(df["away_id"]))
    n_teams = len(ids)
    # IDs werden direkt als Zeilenindex in S benutzt -> muessen 0..n_teams-1 sein.
    assert ids == list(range(n_teams)), "Team-IDs sind nicht 0..n-1 (dann Remapping noetig)."

    team_names = {}
    schedule = []
    for _, day in df.groupby("matchday", sort=True):
        pairs = []
        for h, a, hn, an in zip(day["home_id"], day["away_id"],
                                day["home_name"], day["away_name"]):
            h, a = int(h), int(a)
            team_names[h], team_names[a] = hn, an
            pairs.append((h, a))
        schedule.append(pairs)
    return schedule, team_names, n_teams

schedule, team_names, N_TEAMS = load_schedule(SCHEDULE_PATH)
n_matchdays = len(schedule)
print(f"Spielplan: {N_TEAMS} Teams, {n_matchdays} Spieltage, "
      f"{sum(len(d) for d in schedule)} Partien")

Spielplan: 20 Teams, 38 Spieltage, 380 Partien


## 3. Tordifferenz & Poisson-Parameter

$TD = S_i - S_j$ am neutralen Platz (kein Heimvorteil). Daraus die Poisson-Parameter
$\lambda_i=\tfrac12(\Sigma+TD)$, $\lambda_j=\tfrac12(\Sigma-TD)$; negative Werte werden auf 0,1 gesetzt.

In [9]:
def goal_diff(S, home, away, m):
    """TD = S_i - S_j am neutralen Platz. Kein Heimvorteil."""
    return S[home, m] - S[away, m]

def goals_to_lambda(td, Sigma):
    """lam_i = 1/2 (Sigma + td),  lam_j = 1/2 (Sigma - td).
    Negatives lambda (nur falls |td| > Sigma) -> LAMBDA_FLOOR."""
    lam_i = 0.5 * (Sigma + td)
    lam_j = 0.5 * (Sigma - td)
    # Abfangregel.
    lam_i = LAMBDA_FLOOR if lam_i < 0.0 else lam_i
    lam_j = LAMBDA_FLOOR if lam_j < 0.0 else lam_j
    return lam_i, lam_j

## 4. Zufallseffekte & Saison → `long_df`

Tore unabhängig Poisson-verteilt ziehen. `simulate_season_long` erzeugt **direkt** das
long_df-Schema: **zwei Zeilen je Spiel** (Heim- und Auswärtsperspektive) mit
$X_\text{home} = -X_\text{away}$. In der Simulation ist kein Heimvorteil eingebaut,
daher ist `X` die rohe Tordifferenz aus Sicht von `team_id` (keine 2h-Bereinigung nötig).

In [10]:
def draw_goals(lam_i, lam_j, rng):
    """T_i ~ Poi(lam_i), T_j ~ Poi(lam_j), unabhaengig gezogen."""
    return int(rng.poisson(lam_i)), int(rng.poisson(lam_j))

def simulate_season_long(S, schedule, Sigma, rng, season_id, base_date):
    """Simuliert eine Saison und liefert sie direkt im long_df-Format.
    Zwei Zeilen je Spiel (Heim-/Auswaertsperspektive), X_home = -X_away.
    Spalten: season_id, date, team_id, opponent_id, X, is_home, match_number.
    match_number = m (jedes Team spielt je Spieltag genau einmal)."""
    rows = []
    for m, day in enumerate(schedule):
        date = base_date + pd.Timedelta(days=m)          # monotoner Platzhalter je Team
        for home, away in day:
            td = goal_diff(S, home, away, m)
            lam_i, lam_j = goals_to_lambda(td, Sigma)
            g_home, g_away = draw_goals(lam_i, lam_j, rng)
            x_home = float(g_home - g_away)
            rows.append({"season_id": season_id, "date": date, "team_id": home,
                         "opponent_id": away, "X": x_home, "is_home": True,
                         "match_number": m})
            rows.append({"season_id": season_id, "date": date, "team_id": away,
                         "opponent_id": home, "X": -x_home, "is_home": False,
                         "match_number": m})
    return pd.DataFrame(rows)

## 5. Simulation ausführen

Stärke-Modell wählen (`"constant"`, `"random"`, `"ar1"`oder `"const_form"`). Pro Saison eine neue
Stärke-Realisierung (Sommerpause ändert die Dynamik); der reale
Spielplan wird über die Saisons wiederverwendet. Sortierung wie im long_df-Schema.

**Aufbau dieses Abschnitts — wichtig, weil die Simulation auto-geseedet ist:**
Ein erneuter Lauf erzeugt *andere* Zahlen, würde also die Datensätze ersetzen, auf denen
die Auswertung in `03_acf/` bereits beruht. Deshalb:

1. **Definitions-Zelle** (direkt unten) — `simulate_strength`, `generate_dataset`,
   `dataset_filename`, `OUT`. Schreibt **keine** Datei und kann jederzeit gefahrlos
   ausgeführt werden; sie ist Voraussetzung für alle Generierungs-Zellen.
2. **Generierungs-Zellen** (31 Saisons, 300 Saisons, AR(1)-`alpha`-Reihe) — jede schreibt
   CSVs und hat oben ein `OVERWRITE`-Flag. Mit `OVERWRITE = False` (Default) wird eine
   bereits vorhandene Datei **übersprungen**, statt sie neu zu würfeln; nur mit
   `OVERWRITE = True` wird sie ersetzt.

Damit ist auch "Run All" unkritisch: neue Konfigurationen (z.B. weitere `alpha`-Werte)
werden erzeugt, alles Vorhandene bleibt unverändert.

In [11]:
# === Nur Definitionen -- diese Zelle schreibt KEINE Datei ================
# Bewusst getrennt von den Generierungs-Zellen darunter: die Definitionen werden fuer
# JEDEN Lauf gebraucht (auch fuer die alpha-Zelle weiter unten), duerfen also gefahrlos
# ausgefuehrt werden, ohne vorhandene CSVs neu zu wuerfeln.

def simulate_strength(model, n_teams, n_matchdays, rng, alpha=None, rho=None):
    if alpha is None: alpha = ALPHA
    if rho   is None: rho   = RHO
    if model == "constant":
        return simulate_strength_constant(n_teams, n_matchdays, V, rng)
    if model == "random":
        return simulate_strength_random(n_teams, n_matchdays, V, rng)
    if model == "ar1":
        return simulate_strength_ar1(n_teams, n_matchdays, V, alpha, rng)
    if model == "const_form":
        return simulate_strength_constant_plus_form(n_teams, n_matchdays, V, rho, rng)
    raise ValueError(f"Unbekanntes Modell: {model}")

def generate_dataset(model, n_seasons, schedule, alpha=None, rho=None):
    """Erzeugt einen Datensatz im long_df-Format. Auto-geseedet (kein fester Seed):
    frische OS-Entropie je Aufruf -> Datensaetze sind untereinander unabhaengig."""
    if alpha is None: alpha = ALPHA
    if rho   is None: rho   = RHO
    rng = np.random.default_rng()                      # kein Seed-Argument
    frames = []
    for s in range(n_seasons):
        S = simulate_strength(model, N_TEAMS, n_matchdays, rng, alpha, rho)
        frames.append(simulate_season_long(S, schedule, SIGMA, rng,
                                            f"{model}_{s}", BASE_DATE))
    df = pd.concat(frames, ignore_index=True)
    return df.sort_values(["season_id", "team_id", "date"],
                          kind="mergesort").reset_index(drop=True)

def dataset_filename(model, n_seasons, alpha):
    """Einheitliches Namensschema; alpha steht nur beim AR(1) im Namen."""
    return (f"{model}_{n_seasons}seasons_alpha{alpha:.2f}.csv" if model == "ar1"
            else f"{model}_{n_seasons}seasons.csv")

OUT = REPO_ROOT / "data" / "simulated"
OUT.mkdir(parents=True, exist_ok=True)
print(f"Definitionen geladen. Ausgabeordner: {OUT}")
print("Vorhanden:", ", ".join(sorted(p.name for p in OUT.glob('*seasons*.csv'))) or "(leer)")

Definitionen geladen. Ausgabeordner: /Users/piet/Desktop/6. Mathe/Bachelorarbeit/Code/bachelorarbeit/data/simulated
Vorhanden: ar1_300seasons_alpha0.90.csv, ar1_31seasons_alpha0.90.csv, const_form_300seasons.csv, const_form_31seasons.csv, constant_300seasons.csv, constant_31seasons.csv, random_300seasons.csv, random_31seasons.csv


In [12]:
# === Alle vier Modelle auf einmal; Größe des realen Datensatzes ==========
N_SEASONS = 31
ALPHA     = 0.9
OVERWRITE = False        # False: vorhandene CSVs bleiben unberuehrt (auto-geseedet ->
                         # ein neuer Lauf liefert ANDERE Zahlen als die in 03_acf/ ausgewerteten)

for model in ["constant", "random", "const_form", "ar1"]:
    fname = dataset_filename(model, N_SEASONS, ALPHA)
    path  = OUT / fname
    if path.exists() and not OVERWRITE:                # vor der teuren Simulation pruefen
        print(f"uebersprungen (existiert bereits): {fname}")
        continue
    df = generate_dataset(model, N_SEASONS, schedule)
    df.to_csv(path, index=False)
    print(f"{model}: {len(df)} Zeilen ({len(df)//2} Spiele) -> {fname}")

uebersprungen (existiert bereits): constant_31seasons.csv
uebersprungen (existiert bereits): random_31seasons.csv
uebersprungen (existiert bereits): const_form_31seasons.csv
uebersprungen (existiert bereits): ar1_31seasons_alpha0.90.csv


In [13]:
# === Dieselben vier Modelle mit 300 Saisons (kleinere Fehlerbalken) ======
N_SEASONS = 300
ALPHA     = 0.9
OVERWRITE = False        # siehe Zelle darueber: schuetzt die vorhandenen CSVs

for model in ["constant", "random", "const_form", "ar1"]:
    fname = dataset_filename(model, N_SEASONS, ALPHA)
    path  = OUT / fname
    if path.exists() and not OVERWRITE:                # vor der teuren Simulation pruefen
        print(f"uebersprungen (existiert bereits): {fname}")
        continue
    df = generate_dataset(model, N_SEASONS, schedule)
    df.to_csv(path, index=False)
    print(f"{model}: {len(df)} Zeilen ({len(df)//2} Spiele) -> {fname}")

uebersprungen (existiert bereits): constant_300seasons.csv
uebersprungen (existiert bereits): random_300seasons.csv
uebersprungen (existiert bereits): const_form_300seasons.csv
uebersprungen (existiert bereits): ar1_300seasons_alpha0.90.csv


### AR(1) mit weiteren `alpha`-Werten

Die Zelle unten erzeugt fuer jedes `alpha` in `ALPHAS` einen eigenen 31-Saisons-Datensatz.

`OVERWRITE = False` schuetzt bereits vorhandene Dateien: die Simulation ist auto-geseedet,
ein erneuter Lauf wuerde also andere Zahlen liefern als die, die in `03_acf/` bereits
ausgewertet sind. Zum bewussten Neuwuerfeln `OVERWRITE = True` setzen.

In [18]:
# === AR(1) mit frei waehlbaren alphas ====================================
# Schreibt NUR ar1-Dateien, deren alpha im Namen steht -- die Datensaetze der anderen
# Modelle (constant/random/const_form) werden hier nie angefasst.
# Nur diese drei Zeilen anpassen:
ALPHAS        = [np.exp(-1 / 7), 0.50, 0.97, 0.30]   # <<< alphas beliebig setzen (0 < alpha < 1)
N_SEASONS_AR1 = 300                             # Saisons je Datensatz
OVERWRITE     = False                          # True = vorhandene Datei neu wuerfeln

assert all(0.0 < a < 1.0 for a in ALPHAS), "alpha muss in (0,1) liegen (Stationaritaet, tau endlich)"

# Der Dateiname rundet alpha auf zwei Stellen -> die alphas muessen sich darin unterscheiden,
# sonst wuerde ein Lauf den anderen ueberschreiben.
fnames = [dataset_filename("ar1", N_SEASONS_AR1, a) for a in ALPHAS]
assert len(set(fnames)) == len(fnames), f"Dateinamen kollidieren (alpha auf 2 Stellen): {fnames}"

for alpha, fname in zip(ALPHAS, fnames):
    tau  = -1.0 / np.log(alpha)                # Abklingzeit in Spieltagen (Soll fuer den freien ACF-Fit)
    path = OUT / fname
    info = (f"alpha={alpha:.4f}  tau={tau:5.2f} Spieltage  "
            f"(Halbwertszeit {tau * np.log(2):4.1f},  sigma_eps={ar1_sigma_eps(V, alpha):.4f})")
    if path.exists() and not OVERWRITE:         # bestehende Laeufe nicht versehentlich neu wuerfeln
        print(f"uebersprungen (existiert bereits): {fname}   {info}")
        continue
    df = generate_dataset("ar1", N_SEASONS_AR1, schedule, alpha=alpha)
    df.to_csv(path, index=False)
    print(f"{fname}   {info}   [{len(df)} Zeilen, {len(df) // 2} Spiele]")

uebersprungen (existiert bereits): ar1_300seasons_alpha0.87.csv   alpha=0.8669  tau= 7.00 Spieltage  (Halbwertszeit  4.9,  sigma_eps=0.3550)
uebersprungen (existiert bereits): ar1_300seasons_alpha0.50.csv   alpha=0.5000  tau= 1.44 Spieltage  (Halbwertszeit  1.0,  sigma_eps=0.6167)
uebersprungen (existiert bereits): ar1_300seasons_alpha0.97.csv   alpha=0.9700  tau=32.83 Spieltage  (Halbwertszeit 22.8,  sigma_eps=0.1731)
ar1_300seasons_alpha0.30.csv   alpha=0.3000  tau= 0.83 Spieltage  (Halbwertszeit  0.6,  sigma_eps=0.6793)   [228000 Zeilen, 114000 Spiele]
